# 4 · Did the models learn ripening or the season?

The only annotated run (series 1) lasted from 30 March to 2 August 2023. The sausages ripened in a traditional attic
without climate control, so **temperature rose together with the ripening phase**. A model can then reach a high score
by recognising "it is summer" instead of "the sausage is ripe". The score on series 1 cannot reveal this, because there
season and phase always go together.

The unannotated **series 2** (31 July 2023 to 3 April 2024: summer → winter → spring) can, even without labels:
ripening only moves forward, so the predicted phase must rise over time. A temperature-driven model predicts late
phases in August and falls back to early ones in winter.

This notebook trains every model with each feature set (`FEATURE_SETS` in `ml_core/training.py`) and compares
* **series 1, balanced accuracy**: leave-one-sausage-out, as in notebook 3;
* **series 2, phase order ρ**: rank correlation of time and predicted phase (+1 = always advancing, negative = backwards);
* **series 2, backwards share**: share of time the predicted phase lies below one reached earlier.

Caveat: series 2 may also ripen at a different pace (cold winter), so ρ says whether phases advance, not whether they are timed right.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))  # project root, so ml_core is importable
import matplotlib.pyplot as plt
import pandas as pd
from ml_core import data
from ml_core.training import FEATURE_SETS, compare_feature_sets, load_windows
pd.set_option("display.width", 200)

In [ ]:
CONVERTED_DIR = data.CONVERTED_DIR
WINDOW_SIZE, STRIDE = 10, 2
FEATURE_SETS_TO_COMPARE = list(FEATURE_SETS)
QUICK = False

In [ ]:
windows = load_windows(CONVERTED_DIR, window_size=WINDOW_SIZE, stride=STRIDE)
unannotated = load_windows(CONVERTED_DIR, "series_2_*.csv", WINDOW_SIZE, STRIDE, annotated=False, classes=windows.classes)
pd.concat([windows.summary(), unannotated.summary()])[["recording", "windows", "from", "to"]]

## Temperature over both runs: the confound in one picture

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3))
for w, colour in ((windows, "tab:red"), (unannotated, "tab:blue")):
    for rec, raw in w.raw.items():
        t = raw[raw["sensor"] == "ltcfs"]
        ax.plot(pd.to_datetime(t["timestamp"]), t["ltcfs_temperature"].rolling(50, min_periods=1).median(), color=colour, lw=.6, alpha=.6)
ax.set(ylabel="LTC-FS temperature (°C)", title="red: series 1 (annotated) · blue: series 2 (unannotated)")
plt.show()

In [ ]:
table, progressions = compare_feature_sets(windows, unannotated, FEATURE_SETS_TO_COMPARE, quick=QUICK)
table

## Predicted phase on series 2 per feature set (smoothed over one day)

In [ ]:
fsets = FEATURE_SETS_TO_COMPARE
keys = sorted({k for _, k in progressions}, key=["logreg", "rf", "mlp"].index)
fig, axes = plt.subplots(len(fsets), len(keys), figsize=(15, 2.6 * len(fsets)), sharex=True, sharey=True, squeeze=False)
for i, fs in enumerate(fsets):
    for j, key in enumerate(keys):
        ax = axes[i][j]
        for rec, s in progressions[(fs, key)].items():
            ax.plot(s.index, s.values, lw=1)
        ax.set(title=f"{fs} · {key}", yticks=range(len(windows.classes)), yticklabels=windows.classes)
        ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

**How to read it:** a feature set is only trustworthy if it scores well on series 1 *and* advances on series 2.
The default for notebooks 2 and 3 is `DEFAULT_FEATURE_SET` in `ml_core/training.py`. Change it there if the results
favour another set, then re-run notebooks 2 and 3.